# make_notifications

**Simulate a live feed of experiment updates** by writing a tiny marker into the metadata of recently-run `pdv_alpss_result` files, one at a time, spaced out over time.

Each metadata write triggers the server's `item.updated` announcement, so walking the recent files oldest→newest with a pause between each produces a realistic stream of "new update" events for anything listening on the live push channel.

Steps:
1. Authenticate a Girder client against `https://data.htmdec.org/api/v1`.
2. Use the `aimdl` endpoints to collect every `pdv_alpss_result` file whose **experiment date** falls in the last seven days.
3. Starting with the oldest file, add `notification_test: yes` to each, pausing **20 seconds** between writes.
4. Clean up: remove the `notification_test` key from every touched file (all at once) so nothing is left changed.

> The marker is a single throwaway key and the final cell fully reverts it — no experimental data is modified.

## Setup

In [1]:
import json
import sys
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

# .env sets GIRDER_API_URL=https://data.htmdec.org/api/v1 and GIRDER_API_KEY
load_dotenv(str(Path.cwd().parent / ".env"))

sys.path.insert(0, str(Path.cwd().parent / "src"))
from aimdl_examples import get_girder_client

# --- knobs ---------------------------------------------------------------
DATA_TYPE = "pdv_alpss_result"
WINDOW_DAYS = 7            # experiment-date window to consider
NOTIFICATION_KEY = "notification_test"
NOTIFICATION_VALUE = "yes"
PAUSE_SECONDS = 5        # gap between successive metadata writes

# Durable, append-only record of every file we mark. Written as each write
# happens, so the markers can be removed even if the kernel dies mid-run, the
# notebook is reopened fresh, or the in-memory list is lost. The server does not
# allow querying items by this key, so this file is the source of truth for undo.
TOUCHED_LOG = Path.cwd() / "make_notifications_touched.jsonl"


def parse_experiment_ts(value):
    """meta.experiment_date -> tz-aware UTC Timestamp (NaT if absent)."""
    if value is None or value == "":
        return pd.NaT
    return pd.to_datetime(value, utc=True, errors="coerce", format="ISO8601")

## 1. Authenticate

`get_girder_client` reads `GIRDER_API_URL` / `GIRDER_API_KEY` from the environment. A single long-lived `requests.Session` is kept open for the whole notebook so it survives the 20-second pauses between writes.

In [2]:
session = requests.Session()
gc = get_girder_client(session=session)

me = gc.get("user/me")
print(f"API:  {gc.urlBase}")
print(f"User: {me['login']} ({me.get('firstName', '')} {me.get('lastName', '')})".rstrip())

API:  https://data.htmdec.org/api/v1/
User: elbert (David Elbert)


## 2. Collect recent files (experiment date in the last 7 days)

`aimdl/partition` returns `{"IGSN//experiment_date": checksum}` for the data type. We keep the partitions whose date is within the window, then resolve each partition's IGSN to its actual file items via `aimdl/datafiles` (which carry the `_id` needed to write metadata). Items are re-checked against the window on `meta.experiment_date` and sorted oldest→newest by (experiment date, created time).

In [3]:
cutoff = datetime.now(timezone.utc) - timedelta(days=WINDOW_DAYS)
print(f"Keeping experiment dates >= {cutoff.isoformat()}\n")

# --- partitions updated within the window -------------------------------
partitions = gc.get("aimdl/partition", parameters={"dataType": DATA_TYPE})
recent_igsns = sorted({
    key.split("//")[0]
    for key in partitions
    if parse_experiment_ts(key.split("//")[1]) >= cutoff
})
print(f"{len(recent_igsns)} IGSN partition(s) in the last {WINDOW_DAYS} days:")
print("  " + ", ".join(recent_igsns))


def igsn_items(igsn):
    """Every DATA_TYPE item for one IGSN, paged, with meta.experiment_date."""
    out, offset = [], 0
    while True:
        batch = gc.get(
            "aimdl/datafiles",
            parameters={
                "dataType": DATA_TYPE,
                "limit": 50,
                "offset": offset,
                "filters": json.dumps({"meta.igsn": igsn}),
                "extraFields": '["meta.experiment_date"]',
            },
        )
        out += batch
        if len(batch) < 50:
            return out
        offset += 50


# --- resolve partitions to individual files, re-filtered on experiment date ---
rows = []
for igsn in recent_igsns:
    for item in igsn_items(igsn):
        exp_ts = parse_experiment_ts(item.get("meta", {}).get("experiment_date"))
        if pd.isna(exp_ts) or exp_ts < cutoff:
            continue
        rows.append({
            "itemId": item["_id"],
            "name": item["name"],
            "igsn": igsn,
            "experiment_date": exp_ts,
            "created": pd.to_datetime(item["created"], utc=True, errors="coerce"),
        })

files = (
    pd.DataFrame(rows)
    .drop_duplicates(subset="itemId")
    .sort_values(["experiment_date", "created"])
    .reset_index(drop=True)
)

est_minutes = max(0, len(files) - 1) * PAUSE_SECONDS / 60
print(f"\n{len(files)} file(s) to update.")
print(f"Estimated run time for step 3: ~{est_minutes:.1f} min "
      f"({len(files)} writes, {PAUSE_SECONDS}s apart).")
files[["experiment_date", "created", "igsn", "name", "itemId"]]

Keeping experiment dates >= 2026-07-05T02:06:40.825575+00:00

9 IGSN partition(s) in the last 7 days:
  JHACRA00004-004, JHAMAL00016-014, JHAMAL00016-016, JHAMAL00016-017, JHAMAL00018-017, JHAMAL00018-018, JHAMAL00018-021, NWXMAB00009-002-001, NWXMAB00010-002-001

79 file(s) to update.
Estimated run time for step 3: ~6.5 min (79 writes, 5s apart).


,experiment_date,created,igsn,name,itemId
0,2026-07-06 00:00:00+00:00,2026-07-06 18:04:48.545000+00:00,JHACRA00004-004,C1--JHACRA00004-004_2026-07-06_18-03-00_shot01...,6a4bee40e95d6374345f579f
1,2026-07-06 00:00:00+00:00,2026-07-06 18:04:52.417000+00:00,JHACRA00004-004,C2--JHACRA00004-004_2026-07-06_18-03-00_shot01...,6a4bee44e95d6374345f57bb
2,2026-07-06 00:00:00+00:00,2026-07-06 18:04:55.916000+00:00,JHACRA00004-004,C3--JHACRA00004-004_2026-07-06_18-03-00_shot01...,6a4bee47e99a37bb69a4befc
3,2026-07-06 00:00:00+00:00,2026-07-06 18:04:59.544000+00:00,JHACRA00004-004,C1--JHACRA00004-004_2026-07-06_18-03-10_shot02...,6a4bee4b529e4df6c516bd51
4,2026-07-06 00:00:00+00:00,2026-07-06 18:05:02.801000+00:00,JHACRA00004-004,C2--JHACRA00004-004_2026-07-06_18-03-10_shot02...,6a4bee4ee99a37bb69a4bf18
...,...,...,...,...,...
74,2026-07-10 00:00:00+00:00,2026-07-10 15:52:45.991000+00:00,JHAMAL00016-017,C1--JHAMAL00016-017_69d3f8af8daaca38a7aa7f7d_2...,6a51154dc4f7cfd858c89a37
75,2026-07-10 00:00:00+00:00,2026-07-10 15:53:52.004000+00:00,JHAMAL00016-017,C1--JHAMAL00016-017_69d3f8af8daaca38a7aa7f7d_2...,6a511590c1f01f704abeb274
76,2026-07-10 00:00:00+00:00,2026-07-10 15:54:48.653000+00:00,JHAMAL00016-017,C1--JHAMAL00016-017_69d3f8af8daaca38a7aa7f7d_2...,6a5115c8c4f7cfd858c89aac
77,2026-07-10 00:00:00+00:00,2026-07-10 15:54:51.700000+00:00,JHAMAL00016-017,C1--JHAMAL00016-017_69d3f8af8daaca38a7aa7f7d_2...,6a5115cbc4f7cfd858c89ac8


## 3. Add `notification_test` metadata, oldest → newest

One write per file, `PAUSE_SECONDS` between them, to simulate updates arriving over time. `addMetadataToItem` merges — it only adds the marker key and leaves all existing metadata intact. The touched item IDs are recorded so step 4 can revert exactly what was changed.

> This cell runs for roughly the estimated time above. Interrupting it mid-run is safe — step 4 reverts whatever was written so far.

In [4]:
touched_item_ids = []

n = len(files)
for i, row in files.iterrows():
    gc.addMetadataToItem(row["itemId"], {NOTIFICATION_KEY: NOTIFICATION_VALUE})
    touched_item_ids.append(row["itemId"])

    # Persist immediately, before the pause, so an interrupt still leaves a
    # complete record of everything written so far.
    with open(TOUCHED_LOG, "a") as fh:
        fh.write(json.dumps({"itemId": row["itemId"], "name": row["name"]}) + "\n")

    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"[{stamp}] {i + 1}/{n}  {NOTIFICATION_KEY}={NOTIFICATION_VALUE!r}  "
          f"({row['experiment_date'].date()})  {row['name']}")

    if i < n - 1:  # no pause after the last write
        time.sleep(PAUSE_SECONDS)

print(f"\nDone. Added '{NOTIFICATION_KEY}' to {len(touched_item_ids)} file(s).")
print(f"Recorded in {TOUCHED_LOG}")

[02:06:48] 1/79  notification_test='yes'  (2026-07-06)  C1--JHACRA00004-004_2026-07-06_18-03-00_shot01--00000-results.csv
[02:06:53] 2/79  notification_test='yes'  (2026-07-06)  C2--JHACRA00004-004_2026-07-06_18-03-00_shot01--00000-results.csv
[02:06:58] 3/79  notification_test='yes'  (2026-07-06)  C3--JHACRA00004-004_2026-07-06_18-03-00_shot01--00000-results.csv
[02:07:03] 4/79  notification_test='yes'  (2026-07-06)  C1--JHACRA00004-004_2026-07-06_18-03-10_shot02--00000-results.csv
[02:07:08] 5/79  notification_test='yes'  (2026-07-06)  C2--JHACRA00004-004_2026-07-06_18-03-10_shot02--00000-results.csv
[02:07:13] 6/79  notification_test='yes'  (2026-07-06)  C3--JHACRA00004-004_2026-07-06_18-03-10_shot02--00000-results.csv
[02:07:18] 7/79  notification_test='yes'  (2026-07-06)  C1--JHACRA00004-004_2026-07-06_18-03-20_shot03--00000-results.csv
[02:07:23] 8/79  notification_test='yes'  (2026-07-06)  C2--JHACRA00004-004_2026-07-06_18-03-20_shot03--00000-results.csv
[02:07:28] 9/79  notific

## 4. Clean up — remove the marker (all at once)

Deletes the `notification_test` key from every touched file via `DELETE item/{id}/metadata`, returning each file to its original state. No pauses here.

In [8]:
if not touched_item_ids:
    print("Nothing to clean up — no files were touched in step 3.")
else:
    for item_id in touched_item_ids:
        gc.sendRestRequest("DELETE", f"item/{item_id}/metadata", json=[NOTIFICATION_KEY])
    print(f"Removed '{NOTIFICATION_KEY}' from {len(touched_item_ids)} file(s). "
          "Files restored to original state.")

    # Optional verification: confirm the key is gone from the first touched item.
    check = gc.get(f"item/{touched_item_ids[0]}")
    still_present = NOTIFICATION_KEY in check.get("meta", {})
    print(f"Verify first item still has '{NOTIFICATION_KEY}': {still_present}")

Removed 'notification_test' from 79 file(s). Files restored to original state.
Verify first item still has 'notification_test': False


## Safety cleanup — undo from the durable record (kernel-independent)

The step-4 cell above is the convenient in-session undo, but it depends on the `touched_item_ids` list living in memory. If the kernel died mid-run, you restarted, or you reopened this notebook later, that list is gone.

This cell instead reads `make_notifications_touched.jsonl` — the append-only log written during step 3 — and removes the marker from every file listed there, independent of any in-memory state. **To recover after a crash: run only the *Setup* and *Authenticate* cells, then run this one.** It is idempotent (files already clean are skipped) and safe to run repeatedly.

In [6]:
if not TOUCHED_LOG.exists():
    print(f"No log at {TOUCHED_LOG} -- nothing recorded to clean up.")
else:
    records = [json.loads(line) for line in TOUCHED_LOG.read_text().splitlines() if line.strip()]
    ids = list(dict.fromkeys(r["itemId"] for r in records))  # dedup, preserve order
    print(f"{len(ids)} unique file(s) recorded in {TOUCHED_LOG.name}.")

    removed, already_clean, errors = 0, 0, 0
    for item_id in ids:
        try:
            meta = gc.get(f"item/{item_id}").get("meta", {})
            if NOTIFICATION_KEY in meta:
                gc.sendRestRequest("DELETE", f"item/{item_id}/metadata", json=[NOTIFICATION_KEY])
                removed += 1
            else:
                already_clean += 1
        except Exception as e:
            errors += 1
            print(f"  ! {item_id}: {e}")

    print(f"Removed marker from {removed} file(s); {already_clean} already clean; {errors} error(s).")
    if errors == 0:
        print("All recorded files are clean.")
        # Retire the log so it can't be replayed later (optional):
        # TOUCHED_LOG.unlink()

79 unique file(s) recorded in make_notifications_touched.jsonl.
Removed marker from 0 file(s); 79 already clean; 0 error(s).
All recorded files are clean.


### Last-resort re-scan (only if the log file itself is lost)

If `make_notifications_touched.jsonl` was deleted or written to a different working directory, the marker can't be looked up server-side (the key isn't a queryable field). This cell falls back to **re-deriving the candidate files** by re-walking recent partitions and stripping the marker from any item that still carries it. Widen `FALLBACK_DAYS` if a run may have straddled older experiment dates. Requires the *Setup* and *Authenticate* cells.

In [7]:
FALLBACK_DAYS = 30  # widen the experiment-date window to be safe
fb_cutoff = datetime.now(timezone.utc) - timedelta(days=FALLBACK_DAYS)

parts = gc.get("aimdl/partition", parameters={"dataType": DATA_TYPE})
igsns = sorted({
    k.split("//")[0] for k in parts
    if parse_experiment_ts(k.split("//")[1]) >= fb_cutoff
})

scanned, removed = 0, 0
for igsn in igsns:
    offset = 0
    while True:
        batch = gc.get("aimdl/datafiles", parameters={
            "dataType": DATA_TYPE, "limit": 50, "offset": offset,
            "filters": json.dumps({"meta.igsn": igsn}),
        })
        for item in batch:
            scanned += 1
            # Re-fetch the full item so we see all metadata keys, not just the
            # fields the list view returns.
            meta = gc.get(f"item/{item['_id']}").get("meta", {})
            if NOTIFICATION_KEY in meta:
                gc.sendRestRequest("DELETE", f"item/{item['_id']}/metadata", json=[NOTIFICATION_KEY])
                removed += 1
                print(f"  cleared {item['name']}")
        if len(batch) < 50:
            break
        offset += 50

print(f"Scanned {scanned} file(s) across {len(igsns)} recent IGSN(s); "
      f"removed marker from {removed}.")

Scanned 520 file(s) across 17 recent IGSN(s); removed marker from 0.
